# Reproduce Experiment 6 — black hole entropy under absorption

Retrieves job `d7t6lj7ljm6s73bc82c0` (145 PUBs: 18 for Part A, 7 × 16 for Part B, 15 for an empty baseline circuit). It recomputes, from hardware data, the entropy S(t) of Part A, the two-qubit entropy S(θ) and ΔA = 4ΔS of Part B, and the energy response Δ⟨H_SYK⟩. The simulation reference values come from the Trotterized simulation in the experiment notebook. **Result:** the energy response matches simulation; the area growth ΔA(θ) is not resolved on hardware.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
import numpy as np

service = QiskitRuntimeService(channel="ibm_cloud")

def check(name, value, paper, tol):
    ok = abs(value - paper) <= tol
    print(f"{name:32s} recomputed {value:9.4f}   paper {paper:9.4f}   {'OK' if ok else 'DIFFERENT'}")

In [3]:
from itertools import product

I2 = np.eye(2); X = np.array([[0, 1], [1, 0]]); Y = np.array([[0, -1j], [1j, 0]]); Z = np.array([[1, 0], [0, -1]])
P = {"I": I2, "X": X, "Y": Y, "Z": Z}

def S_vn(rho):
    ev = np.maximum(np.linalg.eigvalsh(rho), 1e-12)
    return float(-np.sum(ev * np.log2(ev)))

def rho_1q(x, y, z):
    n = np.sqrt(x*x + y*y + z*z)
    if n > 1: x, y, z = x/n, y/n, z/n
    return (I2 + x*X + y*Y + z*Z) / 2

LABELS = ["I" + p1 + p0 for p0, p1 in product("IXYZ", "IXYZ") if not (p0 == "I" and p1 == "I")]

def rho_2q(values):
    rho = np.eye(4, dtype=complex) / 4
    for lab, v in zip(LABELS, values):
        rho += v * np.kron(P[lab[2]], P[lab[1]]) / 4
    return rho

job = service.job("d7t6lj7ljm6s73bc82c0")
r = job.result()
ev = [float(r[i].data.evs) for i in range(145)]

In [4]:
T = [0, 1, 2, 3, 4, 5]
S_A = [S_vn(rho_1q(*ev[3*i:3*i + 3])) for i in range(6)]

THETA = np.linspace(0, np.pi, 7)
S_B, H_B = [], []
for i in range(7):
    block = ev[18 + 16*i: 18 + 16*(i + 1)]
    S_B.append(S_vn(rho_2q(block[:15])))
    H_B.append(block[15])
S_B, H_B = np.array(S_B), np.array(H_B)
dA_hw = 4 * (S_B - S_B[0])
dH_hw = H_B - H_B[0]
S_base = S_vn(rho_2q(ev[130:145]))

SIM_S_A  = [0.0000, 0.0404, 0.1197, 0.2140, 0.3119, 0.4074]
SIM_S_B  = np.array([0.3483, 0.3689, 0.4237, 0.4962, 0.5666, 0.6170, 0.6353])
SIM_dH   = np.array([0.0000, -0.0037, -0.0139, -0.0278, -0.0417, -0.0519, -0.0556])
SIM_dA   = 4 * (SIM_S_B - SIM_S_B[0])

print("Part A   t   S_sim    S_hw")
for t, s0, s1 in zip(T, SIM_S_A, S_A):
    print(f"        {t}   {s0:.4f}   {s1:.4f}")
print("\nPart B   theta/pi   S_sim   S_hw    dA_sim   dA_hw   dH_sim   dH_hw")
for th, a, b, c, d, e, f in zip(THETA/np.pi, SIM_S_B, S_B, SIM_dA, dA_hw, SIM_dH, dH_hw):
    print(f"         {th:.2f}       {a:.3f}   {b:.3f}   {c:6.3f}   {d:6.3f}   {e:7.4f}  {f:7.4f}")
print(f"\nBaseline (empty circuit) S = {S_base:.4f} bits")

Part A   t   S_sim    S_hw
        0   0.0000   0.0055
        1   0.0404   0.3074
        2   0.1197   0.4093
        3   0.2140   0.5017
        4   0.3119   0.6115
        5   0.4074   0.7394

Part B   theta/pi   S_sim   S_hw    dA_sim   dA_hw   dH_sim   dH_hw
         0.00       0.348   0.924    0.000    0.000    0.0000   0.0000
         0.17       0.369   0.965    0.082    0.164   -0.0037  -0.0096
         0.33       0.424   0.971    0.302    0.187   -0.0139  -0.0238
         0.50       0.496   0.922    0.592   -0.010   -0.0278  -0.0478
         0.67       0.567   0.987    0.873    0.252   -0.0417  -0.0667
         0.83       0.617   1.005    1.075    0.324   -0.0519  -0.0795
         1.00       0.635   0.978    1.148    0.214   -0.0556  -0.0906

Baseline (empty circuit) S = 0.0718 bits


In [5]:
check("S_hw(theta=0)", S_B[0], 0.924, 0.0005)
check("S_hw(theta=pi/2)", S_B[3], 0.922, 0.0005)
check("S_hw(theta=pi)", S_B[6], 0.978, 0.0005)
check("dA_hw(theta=pi)", dA_hw[6], 0.214, 0.0005)
check("corr. dS sim/hw", np.corrcoef(SIM_S_B - SIM_S_B[0], S_B - S_B[0])[0, 1], 0.63, 0.005)
check("corr. dH sim/hw", np.corrcoef(SIM_dH, dH_hw)[0, 1], 0.998, 0.0005)

S_hw(theta=0)                    recomputed    0.9243   paper    0.9240   OK
S_hw(theta=pi/2)                 recomputed    0.9219   paper    0.9220   OK
S_hw(theta=pi)                   recomputed    0.9778   paper    0.9780   OK
dA_hw(theta=pi)                  recomputed    0.2140   paper    0.2140   OK
corr. dS sim/hw                  recomputed    0.6276   paper    0.6300   OK
corr. dH sim/hw                  recomputed    0.9981   paper    0.9980   OK
